In [214]:
import os
import polars as pl
from src.bez_bordelu import bez_bordelu
from src.najdi_rok import najdi_rok
from src.pocet_stran import pocet_stran
pl.Config(tbl_rows=500)

In [149]:
df = pl.read_parquet(
    os.path.join("data/cnb_sloupce","100.parquet")
).join(
    pl.read_parquet(os.path.join("data/cnb_sloupce","leader.parquet")), left_on="001", right_on="001", how="left"
).join(
    pl.read_parquet(os.path.join("data/cnb_sloupce","008.parquet")), left_on="001", right_on="001", how="left"
).join(
    pl.read_parquet(os.path.join("data/cnb_sloupce","020.parquet")), left_on="001", right_on="001", how="left"
).join(
    pl.read_parquet(os.path.join("data/cnb_sloupce","022.parquet")), left_on="001", right_on="001", how="left"
).join(
    pl.read_parquet(os.path.join("data/cnb_sloupce","041.parquet")), left_on="001", right_on="001", how="left"
).join(
    pl.read_parquet(os.path.join("data/cnb_sloupce","245.parquet")), left_on="001", right_on="001", how="left"
).join(
    pl.read_parquet(os.path.join("data/cnb_sloupce","260.parquet")), left_on="001", right_on="001", how="left"
).join(
    pl.read_parquet(os.path.join("data/cnb_sloupce","300.parquet")), left_on="001", right_on="001", how="left"
).join(
    pl.read_parquet(os.path.join("data/cnb_sloupce","490.parquet")), left_on="001", right_on="001", how="left"
).explode(
    "300_a"
).with_columns(
    pl.col('300_a').map_elements(pocet_stran, return_dtype=int).alias('stran')
).with_columns(
    pl.col('008').map_elements(najdi_rok, return_dtype=int).alias('rok')
).with_columns(
    pl.col('245_a').map_elements(bez_bordelu, return_dtype=str)
).explode(
    "260_b"
).with_columns(
    pl.col(("260_b")).map_elements(bez_bordelu, return_dtype=str)
).explode(
    "490_a"
).with_columns(
    pl.col(("490_a")).map_elements(bez_bordelu, return_dtype=str)
)

In [127]:
def zjisti(nazev):
    frejm = df.filter(pl.col("245_a").str.contains(nazev)).select(pl.col(["100_a","245_a","260_b","rok","stran","490_a"])).sort(by="rok")
    print(frejm.select(pl.col("490_a")).to_series().to_list()[0])
    return frejm

In [97]:
zjisti("Jak s tím pohnu")

Krystal


100_a,245_a,260_b,rok,stran,490_a
str,str,str,i64,i64,str
"""Houška, Vítězslav,""","""Jak s tím pohnu""","""SNDK""",1965,188,"""Krystal"""


In [75]:
zjisti("Nejkrásnější bývá šílená")

Poesie


100_a,245_a,rok,stran,490_a
str,str,i64,i64,str
"""Cincibuch, Petr,""","""Nejkrásnější bývá šílená""",1983,74,"""Poesie"""
"""Seifert, Jaroslav,""","""Nejkrásnější bývá šílená""",1996,136,"""Květy poezie"""
"""Seifert, Jaroslav,""","""Nejkrásnější bývá šílená""",1968,124,"""Květy poezie"""


In [152]:
zjisti("Pět olympijských")

None


100_a,245_a,260_b,rok,stran,490_a
str,str,str,i64,i64,str
"""Zapletal, Miloš,""","""Pět olympijských kruhů""","""STN""",1964,426,null


In [156]:
def edice(retezec, nakladatelstvi=None):
    if nakladatelstvi == None:
        return df.filter(pl.col("490_a").str.contains(retezec)).select(pl.col(["100_a","245_a","stran","rok"])).sort(by='rok')
    else:
        return df.filter(pl.col("490_a").str.contains(retezec)).filter(pl.col("260_b") == nakladatelstvi).select(pl.col(["100_a","245_a","300_a","rok","stran"])).sort(by='rok')

In [158]:
edice("Škola mladých matematiků")

100_a,245_a,stran,rok
str,str,i64,i64
"""Hradecký, František,""","""Několik úloh z geometrie jedno…",89,1961
"""Sedláček, Jiří,""","""Co víme o přirozených číslech""",42,1961
"""Šisler, Miroslav""","""O funkcích""",56,1962
"""Šedivý, Jaroslav,""","""Shodná zobrazení v konstruktiv…",74,1962
"""Šisler, Miroslav""","""O funkcích""",55,1963
"""Hradecký, František,""","""Několik úloh z geometrie jedno…",92,1963
"""Veselý, František,""","""O nerovnostech""",67,1963
"""Výborný, Rudolf,""","""Matematická indukce""",61,1963
"""Šedivý, Jaroslav,""","""O podobnosti v geometrii""",77,1963


In [55]:
edice("Super lekce")

100_a,245_a,rok,stran
str,str,i64,i64
"""McManus, Sean,""","""Jak se naučit programovat v 10…",2017,64
"""Bishop-Stephens, Will""","""Jak se naučit animovat v 10 sn…",2017,64


In [69]:
edice("AAA : edice anglo-amerických autorů")

100_a,245_a,rok,stran
str,str,i64,i64
"""Erdrich, Louise,""","""Čarování s láskou""",1994,241
"""Kerouac, Jack,""","""Na cestě""",1994,283
"""Toole, John Kennedy,""","""Neónová bible""",1994,120
"""Kosiński, Jerzy N.,""","""Byl jsem při tom""",1995,81
"""Toole, John Kennedy,""","""Spolčení hlupců""",1995,359
"""Brautigan, Richard,""","""Willard a jeho kuželkářské tro…",1995,149
"""Kosiński, Jerzy N.,""","""Nabarvené ptáče""",1995,215
"""Cave, Nick,""","""A uzřela oslice anděla""",1995,297
"""McCarthy, Cormac,""","""Všichni krásní koně""",1995,251


In [77]:
edice("Květy poezie")

100_a,245_a,rok,stran
str,str,i64,i64
"""Goethe, Johann Wolfgang von,""","""Balady""",1958,85
"""Tasso, Torquato,""","""Lyrika""",1958,125
"""Nezval, Vítězslav,""","""Edison""",1958,109
"""Rimbaud, Arthur,""","""Výbor""",1959,175
"""Poe, Edgar Allan,""","""Havran""",1959,89
"""Corbière, Tristan,""","""Žluté lásky""",1959,92
"""Catullus, Gaius Valerius,""","""Básně""",1959,198
"""Dyk, Viktor,""","""Milá sedmi loupežníků""",1959,125
"""Rictus, Jehan,""","""Poezie""",1959,87


In [103]:
edice("Krystal", nakladatelstvi="SNDK")

100_a,245_a,300_a,rok,stran
str,str,list[str],i64,i64
"""Adla, Zdeněk,""","""Krásná a slavná""","[""257, [7] s. :""]",1961,257
"""Bojarová, Olga,""","""Od hlavy k patě""","[""186, [6] s. ;""]",1961,186
"""Petr, Tomislav,""","""Divy oceánů""","[""174 s. :""]",1962,174
"""Koval, Václav,""","""Na dně vzdušného moře""","[""221, [4] s. ;""]",1962,221
"""Štuka, Ivo,""","""Šest dnů na luně 1""","[""200, [3] s. ;""]",1963,200
"""Houška, Vítězslav,""","""Jak s tím pohnu""","[""188 s. ;""]",1965,188
"""Tichý, Jaroslav,""","""Letem ČSSR""","[""214, [6] s. ;""]",1965,214
"""Elstner, František Alexander,""","""Motorové opojení""","[""207, [5] s. ;""]",1966,207
"""Deyl, Václav,""","""Tajemství plamenů""","[""185, [7] s. :""]",1967,185


In [119]:
zjisti("Nová divočina")

ReX


100_a,245_a,260_b,rok,stran,490_a
str,str,str,i64,i64,str
"""Zbořil, Jonáš,""","""Nová divočina""",null,2020,77,"""ReX"""
"""Cook, Diane,""","""Nová divočina""",null,2022,460,"""Světová knihovna"""


In [121]:
edice("ReX")

100_a,245_a,300_a,rok,stran
str,str,list[str],i64,i64
"""Hruška, Petr,""","""Auta vjíždějí do lodí""","[""65 s. :""]",2007,65
"""Slíva, Vít,""","""Souvrať""","[""74 s. :""]",2007,74
"""Malý, Radek,""","""Malá tma""","[""71 s. :""]",2008,71
"""Urbánková, Dagmar,""","""Ta hlava větru vypadá jako pes""","[""107 s. :""]",2008,107
"""Kudláček, Slavomír,""","""Schůzka v lomu""","[""75 s. :""]",2009,75
"""Machulková, Inka,""","""Zamkni les a pojď""","[""85 s. :""]",2009,85
"""Žila, Jaroslav,""","""V hrudi pták""","[""66 s. :""]",2010,66
"""Žila, Jaroslav,""","""V hrudi pták""","[""66 s. :""]",2010,66
"""Reiner, Martin,""","""Hubená stehna Twiggy""","[""103 s. :""]",2010,103


In [133]:
zjisti("Nebe, peklo")

Umělecké snahy


100_a,245_a,260_b,rok,stran,490_a
str,str,str,i64,i64,str
"""Mahen, Jiří,""","""Nebe, peklo, ráj ... (1917)""","""B. Kočí""",1919,103,"""Umělecké snahy"""
"""Mahen, Jiří,""","""Nebe, peklo, ráj ... 1917""","""B. Kočí""",1922,103,"""Umělecké snahy"""
"""Biebl, Konstantin,""","""Nebe, peklo, ráj""","""Erna Janská""",1930,43,"""Editio princeps"""
"""Biebl, Konstantin,""","""Nebe, peklo, ráj""","""Sfinx, Bohumil Janda""",1931,54,"""Růžová zahrada"""
"""Biebl, Konstantin,""","""Nebe, peklo, ráj""",null,1931,54,"""Růžová zahrada"""
"""Mahen, Jiří,""","""Nebe, peklo, ráj... (1917)""",null,1950,139,"""Československé divadelní a lit…"
"""Kornelová, Marie,""","""Nebe, peklo, ráj""","""SNDK""",1967,204,null
"""Cortázar, Julio,""","""Nebe, peklo, ráj""","""Odeon""",1972,580,"""Soudobá světová próza"""
"""Salivarová, Zdena,""","""Nebe, peklo, ráj""","""Atlantis""",1991,271,null


In [135]:
edice('Román', nakladatelstvi="Mladá fronta")

100_a,245_a,300_a,rok,stran
str,str,list[str],i64,i64
"""Kohout, Pavel,""","""Hvězdná hodina vrahů""","[""422 s. ;""]",1995,422
"""Kohout, Pavel,""","""Konec velkých prázdnin""","[""711 s. ;""]",1996,711
"""Kohout, Pavel,""","""Šest & Sex""","[""229 s. ;""]",1998,229
"""Hoffmann, E. T. A.,""","""Ďáblův elixír""","[""291 s. ;""]",2001,291
"""Cortázar, Julio,""","""Nebe, peklo, ráj""","[""600 s. ;""]",2001,600
"""Guillou, Jan,""","""Cesta do Svaté země""","[""356 s. :""]",2001,356
"""Chaviano, Daína,""","""Havana blues""","[""241 s. ;""]",2001,241
"""Aust, Kurt,""","""Den hněvu""","[""356 s. ;""]",2001,356
"""Beckett, Samuel,""","""První láska""","[""71 s. ;""]",2001,71


In [137]:
zjisti("Svobodná")

None


100_a,245_a,260_b,rok,stran,490_a
str,str,str,i64,i64,str
"""Sallač, Karel,""","""Svobodná dělitelnost pozemků j…","""K. Sallač""",1887,33,null
"""Radić, Stjepan,""","""Svobodná škola politických věd…","""Samostatnost""",1899,64,null
"""Doubrava, Josef,""","""Pastýřský list nejdp. biskupa …","""nákl. vl.""",1906,24,"""Časové úvahy"""
"""Černý, Josef,""","""Svobodná, volná škola a její v…",null,1907,187,null
"""Obuchov, Aleksandr Michajlovič…","""Svobodná výchova a disciplina""","""s.n.""",1910,17,"""Sedmá výr. zpr. dívč. lycea sp…"
"""Mach, Jarka,""","""Svobodná""","""K.J. Barvitius""",1934,2,"""Barvitiova edice"""
"""Mach, Jarka,""","""Svobodná""","""K.J. Barvitius""",1934,2,"""Popěvky"""
"""Lamp, Rudolf,""","""Svobodná""","""Státní hudební vydavatelství""",1963,17,"""Zábavní orchestr"""
"""Lamp, Rudolf,""","""Svobodná""","""Státní hudební vydavatelství""",1963,17,"""Salon-orchester"""


In [139]:
zjisti("Pánův hlas")

Omnia


100_a,245_a,260_b,rok,stran,490_a
str,str,str,i64,i64,str
"""Lem, Stanisław,""","""Pánův hlas""","""Svoboda""",1981,729,"""Omnia"""


In [143]:
edice("Omnia", nakladatelstvi="Svoboda")

100_a,245_a,300_a,rok,stran
str,str,list[str],i64,i64
"""Larni, Martti,""","""Ve znamení Panny""","[""206, [4] s. ;""]",1968,206
"""Cronin, A. J.""","""Citadela""","[""446 s. ;""]",1969,446
"""Březovský, Bohuslav,""","""Vzdušné zámky""","[""418, [5] s. ;""]",1969,418
"""Comfort, Will Levington,""","""Zvířata a lidé v džungli""","[""200, [3] s. ;""]",1969,200
"""Dumas, Alexandre,""","""Josef Balsamo""","[""2 sv. (682; 699 s.) :""]",1969,699
"""Wurmser, André,""","""Vrah zemřel první...""","[""135, [1] s. ;""]",1969,135
"""Šmahelová, Helena,""","""Devět tisíc dnů""","[""220, [2] s. ;""]",1969,220
"""Steinbeck, John,""","""Nebeské pastviny""","[""191 s. ;""]",1969,191
"""Mináč, Vladimír,""","""Nikdy nejsi sama""","[""256 s. ;""]",1969,256


In [145]:
zjisti("Ragtime")

None


100_a,245_a,260_b,rok,stran,490_a
str,str,str,i64,i64,str
"""Ghali, Adib""","""Ragtime guitar""","""Levné knihy KMa""",null,45,null
"""Watson, Fr. Th.""","""The Ragtime Girl""",null,1914,4,null
"""Mládek, Ivan,""","""Ragtime-coctail""","""Panton""",1981,13,null
"""Doctorow, E. L.,""","""Ragtime""","""Odeon""",1982,251,"""Soudobá světová próza"""
"""Doctorow, E. L.,""","""Ragtime""","""Odeon""",1985,251,null
"""Mládek, Ivan,""","""Ragtime-coctail""","""Panton""",1985,13,null
"""Joplin, Scott,""","""Original rags. Ragtime""","""Panton""",1988,5,"""Chvilka s kytarou"""
"""Doctorow, E. L.,""","""Ragtime""","""Odeon""",1989,241,null
"""Doctorow, E. L.,""","""Ragtime""","""Levné knihy KMa""",2000,259,"""Edice světových autorů"""


In [147]:
edice("Světová literatura Lidových n")

100_a,245_a,300_a,rok,stran
str,str,list[str],i64,i64
"""Hrabal, Bohumil,""","""Perlička na dně""","[""188 s. ;""]",2005,188
"""García Márquez, Gabriel,""","""Sto roků samoty""","[""318 s. ;""]",2005,318
"""Remarque, Erich Maria,""","""Na západní frontě klid""","[""175 s. ;""]",2005,175
"""Irving, John,""","""Svět podle Garpa""","[""515 s. ;""]",2005,515
"""Škvorecký, Josef,""","""Zbabělci""","[""379 s. ;""]",2005,379
"""McEwan, Ian,""","""Betonová zahrada""","[""127 s. ;""]",2005,127
"""Capote, Truman,""","""Snídaně u Tiffanyho""","[""92 s. ;""]",2005,92
"""Bulgakov, Michail Afanas‘jevič…","""Mistr a Markétka""","[""396 s. ;""]",2005,396
"""Kafka, Franz,""","""Proces""","[""203 s. ;""]",2005,203


In [160]:
zjisti("Jak řešit rovnice a jejich soustavy")

Polytechnická knižnice. Ř. 2, Příručky


100_a,245_a,260_b,rok,stran,490_a
str,str,str,i64,i64,str
"""Jarník, Jiří,""","""Jak řešit rovnice a jejich sou…","""SNTL""",1961,266,"""Polytechnická knižnice. Ř. 2, …"
"""Jarník, Jiří,""","""Jak řešit rovnice a jejich sou…","""SNTL""",1969,243,"""Polytechnická knižnice. Řada 2…"
"""Jarník, Jiří,""","""Jak řešit rovnice a jejich sou…","""SNTL""",1969,243,"""Řada polytechn. literatury"""


In [216]:
edice("Polytechnická knižnice")

100_a,245_a,stran,rok
str,str,i64,i64
"""Sackmann, Horst,""","""Fysikální chemie""",283,1957
"""Kohlmann, Čeněk,""","""Aritmetika a algebra""",228,1958
"""Maška, Otokar,""","""Řešené úlohy z matematiky""",213,1958
"""Košťál, Karel,""","""Sbírka fysikálních vzorců a po…",213,1959
"""Andrlík, Karel,""","""Škola fotografie""",186,1959
"""Krutina, Jaroslav""","""Sbírka vzorců z pružnosti a pe…",254,1959
"""Vavruch, Ivan,""","""Koloidní chemie""",221,1959
"""Vinter, Jan""","""Práce se dřevem ve školních dí…",175,1959
"""Trnka, Jiří,""","""Natíráme sami""",152,1959


In [218]:
zjisti("Normy strojnického kreslení")

Polytechnická knižnice. Řada 1, Technický výběr do kapsy


100_a,245_a,260_b,rok,stran,490_a
str,str,str,i64,i64,str
"""Gregora, Otakar,""","""Normy strojnického kreslení""","""Práce""",1968,226,"""Polytechnická knižnice. Řada 1…"


In [222]:
edice("Technický výběr do kapsy")

100_a,245_a,stran,rok
str,str,i64,i64
"""Sadoul, Georges,""","""Zázraky filmu""",230,1958
"""Dobrovolný, Bohumil,""","""Zopakujme si elektrotechniku""",191,1958
"""Vacek, Adolf""","""Logaritmické tabulky a výpočty""",129,1958
"""Žalud, Karel""","""Rychlosti, vzdálenosti""",175,1958
"""Dobrovolný, Bohumil,""","""Technika v kostce""",199,1958
"""Andrlík, Karel,""","""Zopakujme si chemii""",177,1958
"""Tříska, Jiří,""","""Elektroautomatiky v průmyslu""",138,1958
"""Roček, Vladimír""","""Pokrokové konstrukce obráběcíc…",171,1958
"""Taraba, Oldřich,""","""Co dovede ultrazvuk""",154,1958


In [162]:
edice("Řada polytechn. literatury")

100_a,245_a,stran,rok
str,str,i64,i64
"""Boublík, Vlastimil,""","""Lití plastických hmot pro mode…",199,1966
"""Válek, Jiří""","""Úvod do elektroniky""",195,1966
"""Boublík, Vlastimil,""","""Lepidla a jejich příprava""",190,1966
"""Štěpánský, Václav,""","""Nomogramy""",186,1966
"""Procházka, Arnold""","""Základy mechaniky vody v praxi""",125,1966
"""Fišner, Boleslav,""","""Chemie""",229,1967
"""Sedláček, Jiří,""","""Nebojte se matematiky""",135,1969
"""Jarník, Jiří,""","""Jak řešit rovnice a jejich sou…",243,1969
"""Beránek, Robert,""","""Úprava okolí chaty nebo rodinn…",133,1970


In [168]:
zjisti("Volají neviditelné vlny")

Knížky pro chytré děti


100_a,245_a,260_b,rok,stran,490_a
str,str,str,i64,i64,str
"""Škoda, František,""","""Volají neviditelné vlny""","""SNDK""",1960,47,"""Knížky pro chytré děti"""


In [170]:
edice("Knížky pro chytré děti")

100_a,245_a,stran,rok
str,str,i64,i64
"""Čtvrtek, Václav,""","""Směr vesmír - start!""",44,1959
"""Koval, Václav,""","""Svět za sklem""",47,1959
"""Růžička, Jiří,""","""Nej - nej - nej""",36,1959
"""Korejs, Milan,""","""Rukulíbám - dobrý den""",47,1959
"""Svoboda, Vladimír,""","""Ať přestane svítit""",46,1960
"""Škoda, František,""","""Volají neviditelné vlny""",47,1960
"""Koval, Václav,""","""25 divů v našem domě""",63,1961
"""Souček, Ludvík,""","""Hrátky kolem křižovatky""",54,1962
"""Nepil, František,""","""Kola, strojky, nápady""",69,1963


In [174]:
zjisti("Hry s kalkulátory")

Knižnice všeobecného vzdělání mládeže. Maják


100_a,245_a,260_b,rok,stran,490_a
str,str,str,i64,i64,str
"""Mrázek, Jiří""","""Hry s kalkulátory""","""SPN""",1984,111,"""Knižnice všeobecného vzdělání …"
"""Mrázek, Jiří""","""Hry s kalkulátory""","""SPN""",1986,111,"""Knižnice všeobecného vzděláván…"
"""Mrázek, Jiří""","""Hry s kalkulátory""","""SPN""",1988,111,"""Knižnice všeobecného vzdělání …"


In [178]:
edice("Knižnice všeobecného vzdělání mládeže")

100_a,245_a,stran,rok
str,str,i64,i64
"""Duchoň, František,""","""Hydroponie""",199,1965
"""Hejda, Stanislav,""","""Správná výživa teoreticky a pr…",304,1968
"""Řehák, Bohuslav,""","""Vycházky do přírody""",243,1968
"""Hejda, Stanislav,""","""Správná výživa teoreticky a pr…",304,1969
"""Kuba, Adolf,""","""Kluci, plný plyn""",131,1969
"""Bojev, Sergej Nikolajevič,""","""Jak savci pečují o svá mláďata""",61,1969
"""Šimáček, Radovan,""","""Obchodujeme s celým světem""",264,1970
"""Augusta, Josef,""","""U pravěkých lovců""",163,1971
"""Augusta, Josef,""","""Z hlubin pravěku""",141,1971


In [194]:
df.filter(pl.col("245_a").str.contains("Pájení"))

100_ind1,100_a,100_7,100_4,100_d,100_q,100_c,100_b,100_e,001,leader,008,020_q,020_c,020_a,020_z,022_a,022_y,022_z,022_ind1,022_l,041_ind1,041_a,041_h,041_b,041_k,041_g,041_f,041_d,041_e,041_j,041_n,041_m,245_ind1,245_ind2,245_a,245_b,245_c,245_n,245_p,245_h,245_f,245_s,260_a,260_b,260_c,260_e,260_f,260_g,260_3,260_ind1,300_a,300_b,300_c,300_e,300_f,300_3,490_ind1,490_a,490_v,490_x,490_3,stran,rok
str,str,str,list[str],str,str,list[str],str,str,str,str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,str,str,str,str,str,list[str],list[str],str,str,str,list[str],str,list[str],list[str],list[str],str,list[str],list[str],str,list[str],list[str],list[str],str,str,list[str],str,list[str],list[str],list[str],i64,i64
"""1""","""Ruža, Viliam""","""jx20051114023""","[""aut""]",null,null,null,null,null,"""ck8804457""",""" cam a22 4500""","""880824s1988 xr a f u0…","[""(Váz.) :""]","[""Kčs 40,00""]",null,null,null,null,null,null,null,"""1""","[""cze""]","[""slo""]","[""rus"", ""ger"", ""eng""]",null,null,null,null,null,null,null,null,"""1""","""0""","""Pájení""",null,"""Viliam Ruža ; ze slov. přel. J…",null,null,null,null,null,"[""Praha :"", ""Bratislava :""]","""SNTL""","[""1988""]","[""(Brno :""]","[""Tisk 1)""]",null,null,null,"""452 s. :""","[""obr., fot., tb., grafy ;""]","[""24 cm""]",null,null,null,null,null,null,null,null,452,1988
"""1""","""Ruža, Viliam""","""jx20051114023""","[""aut""]",null,null,null,null,null,"""ck8804457""",""" cam a22 4500""","""880824s1988 xr a f u0…","[""(Váz.) :""]","[""Kčs 40,00""]",null,null,null,null,null,null,null,"""1""","[""cze""]","[""slo""]","[""rus"", ""ger"", ""eng""]",null,null,null,null,null,null,null,null,"""1""","""0""","""Pájení""",null,"""Viliam Ruža ; ze slov. přel. J…",null,null,null,null,null,"[""Praha :"", ""Bratislava :""]","""Alfa""","[""1988""]","[""(Brno :""]","[""Tisk 1)""]",null,null,null,"""452 s. :""","[""obr., fot., tb., grafy ;""]","[""24 cm""]",null,null,null,null,null,null,null,null,452,1988
"""1""","""Martínek, Antonín""",null,"[""aut""]",null,null,null,null,null,"""ck9007228""",""" nam a22 4500""","""910111s1990 xr a u0…","[""(Brož.)""]",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""1""","""0""","""Pájení v elektronice""",null,"""Antonín Martínek""",null,null,null,null,null,"[""Praha :""]","""Tesla-Výzkumný ústav pro sdělo…","[""1990""]",null,null,null,null,null,"""245 s. :""","[""il. ;""]","[""20 cm""]",null,null,null,"[""1""]","""Technické příručky / Tesla, Vý…","[""sv. 34""]",null,null,245,1990
"""1""","""Ruža, Viliam""","""jx20051114023""","[""aut""]",null,null,null,null,null,"""bk197801972""",""" nam a22 1 4500""","""970319s1978 xr e f 0…","[""(Váz.) :""]","[""Kčs 40,00""]",null,null,null,null,null,null,null,"""1""","[""cze""]","[""slo""]",null,null,null,null,null,null,null,null,null,"""1""","""0""","""Pájení""",null,"""Viliam Ruža ; [ze slov. orig. …",null,null,null,null,null,"[""Praha :""]","""SNTL - Nakladatelství technick…","[""1978""]","[""(Brno :""]","[""Tisk 6)""]",null,null,null,"""395 s. :""","[""il., tb. ;""]","[""8°""]",null,null,null,"[""1""]","""Řada strojírenské literatury""",null,null,null,395,1978
"""1""","""Puchnar, Bedřich""","""jx20050531019""","[""edt""]",null,null,null,null,null,"""bk195203804""",""" nam a22 1 4500""","""980511s1952 xr …",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""1""","""0""","""Pájení a svařování, pájky a sv…",null,"""Zprac. B. Puchnar""",null,null,null,null,null,"[""[Praha] :""]","""Min. hutního prům. a rudných d…","[""1952""]","[""([Kladno] :""]","[""SČT 21)""]",null,null,null,"""89, [1] s. ;""",null,"[""8°""]",null,null,null,null,null,null,null,null,89,1952
"""1""","""Němec, Karel""","""jk01083089""","[""aut""]",null,null,null,null,null,"""bk197003834""",""" nam a22 1 4500

In [182]:
zjisti("Současný šperk")

Soudobé české umění


100_a,245_a,260_b,rok,stran,490_a
str,str,str,i64,i64,str
"""Vokáčová, Věra,""","""Současný šperk""","""Odeon""",1979,71,"""Soudobé české umění"""


In [184]:
edice("Soudobé české umění")

100_a,245_a,stran,rok
str,str,i64,i64
"""Hlaváček, Luboš,""","""Současná kresba""",71,1976
"""Vlček, Tomáš,""","""Současný plakát""",71,1976
"""Konečný, Dušan,""","""Výstavy současného umění""",71,1977
"""Mrázová-Schusterová, Marcela,""","""Současná krajinomalba""",71,1977
"""Hlaváček, Luboš,""","""Současná grafika""",72,1977
"""Dvorská, Pavla""","""Současná monumentální tvorba""",71,1978
"""Michl, Jan""","""Realizace a projekty v současn…",71,1978
"""Konečný, Dušan,""","""Mladí čeští malíři""",71,1978
"""Nýdl, Miroslav,""","""Současná známková tvorba""",71,1978


In [198]:
zjisti("Můžu ochutnat")

None


100_a,245_a,260_b,rok,stran,490_a
str,str,str,i64,i64,str
"""Mizielińska, Aleksandra,""","""Můžu ochutnat?""",null,2021,112,null


In [204]:
zjisti(", která se nesmála")

None


100_a,245_a,260_b,rok,stran,490_a
str,str,str,i64,i64,str
"""Tashlin, Frank,""","""Vačice, která se nesmála""","""Baobab""",2013,55,null


In [206]:
zjisti("Nad propastí")

Spisů pro mládež č. 27


100_a,245_a,260_b,rok,stran,490_a
str,str,str,i64,i64,str
"""Vlasák, Josef Věnceslav,""","""Nad propastí""","""Nákladem knihkupectví B. Stýbl…",1882,94,"""Spisů pro mládež č. 27"""
"""Michajlov, A.,""","""Nad propastí""","""A. Hynek""",1883,70,"""Česká bibliotéka rodinná"""
"""Michajlov, A.,""","""Nad propastí""","""Nákladem Al. Hynka, knihkupce""",1884,254,"""Česká bibliotéka rodinná"""
"""Michajlov, A.,""","""Nad propastí""","""Nákladem Al. Hynka, knihkupce""",1884,238,"""Česká bibliotéka rodinná"""
"""Vrchlický, Jaroslav,""","""Nad propastí""",null,1887,46,"""Dramatická díla Jaroslava Vrch…"
"""Zahradník-Brodský, Bohumil,""","""Nad propastí""","""E. Šolc""",1901,371,null
"""Skarlandt, Julius,""","""Nad propastí""","""Edvard Jan Baštýř a spol.""",1908,210,"""Romány Praž. Illustr. kurýra"""
"""Kraszewski, Józef Ignacy,""","""Nad propastí""","""Jos. R. Vilímek""",1913,382,"""Vilímkovy illustrované romány"""
"""Vrchlický, Jaroslav,""","""Nad propastí""","""Nákladem F. Šimáčka""",1913,47,"""Dramatická díla Jaroslava Vrch…"


In [208]:
edice("Crossover")

100_a,245_a,stran,rok
str,str,i64,i64
"""Klein, Naomi,""","""Ne nestačí""",294,2019
"""Luce, Edward,""","""Soumrak západního liberalismu""",191,2019
"""Keen, Andrew""","""Jak opravit budoucnost""",278,2019
"""Ross, Alec,""","""Obory budoucnosti""",330,2019
"""Lee, Kai-fu,""","""Supervelmoci umělé inteligence""",291,2019
"""Lowrey, Annie,""","""Dejte lidem peníze""",251,2020
"""Scheidel, Walter,""","""Velký nivelizátor""",589,2020
"""Collier, Paul,""","""Budoucnost kapitalismu""",299,2020
"""Brown, Archie,""","""Mýtus silného vůdce""",503,2021


In [210]:
zjisti("Neobyvatelná")

Klimax


100_a,245_a,260_b,rok,stran,490_a
str,str,str,i64,i64,str
"""Wallace-Wells, David""","""Neobyvatelná Země""",null,2020,390,"""Klimax"""


In [212]:
edice("Klimax")

100_a,245_a,stran,rok
str,str,i64,i64
"""Wallace-Wells, David""","""Neobyvatelná Země""",390,2020
"""Rich, Nathaniel,""","""Jak ztratit Zemi""",199,2020
"""Hanišová, Viktorie,""","""Beton a hlína""",258,2021
"""Carson, Rachel,""","""Tiché jaro""",374,2021
"""Marshall, George,""","""Ani na to nemyslete""",348,2022
"""Lovelock, James,""","""Novacén""",148,2022
"""Mann, Michael E.,""","""Nová klimatická válka""",444,2022
"""Klein, Naomi,""","""Tím se všechno mění""",702,2023
"""Abram, David,""","""Stávat se zvířetem""",371,2024


In [224]:
zjisti("Každému chléb")

Dobrý vítr


100_a,245_a,260_b,rok,stran,490_a
str,str,str,i64,i64,str
"""Selucký, Radoslav,""","""Každému chléb, každému růže""","""Mladá fronta""",1962,208,"""Dobrý vítr"""


In [226]:
edice("Dobrý vítr")

100_a,245_a,stran,rok
str,str,i64,i64
"""Chrastil, Josef,""","""Smím prosit?""",126,1958
"""Chrastil, Josef,""","""Smím prosit?""",126,1958
"""Majorová, Milena,""","""Člověk mezi lidmi""",123,1958
"""Faukner, Rudolf,""","""Kdyby přišli Marťané, aneb, Pr…",120,1958
"""Dvořák, František""","""Zařizujete si byt?""",83,1958
"""Kusák, Alexej,""","""Knížka o vkusu""",167,1959
"""Boček, Jaroslav,""","""Znáte je z plátna""",183,1959
"""Klímová-Fügnerová, Miroslava,""","""Na prahu života""",91,1959
"""Zahradník, Miroslav,""","""Chcete se líbit?""",96,1959
